# Etapa 4 — Avaliacao offline e golden set

**Datathon 7MLET - Grupo 87**

Mede qualidade tecnica e risco **antes de servir** a decisao:

- Golden set versionado (`data/golden_set/evaluation_cases.jsonl`, >= 20 casos)
- Matriz de metricas bandit (baseline x Thompson Sampling)
- Analise de sensibilidade e fairness de exposicao entre segmentos

> O relatorio oficial e gerado por `poetry run python -m src.evaluation`
> (`reports/offline-evaluation.md`).

In [ ]:
import os
import sys
from pathlib import Path

_p = Path.cwd()
while _p != _p.parent and not (_p / "pyproject.toml").exists():
    _p = _p.parent
PROJECT_ROOT = _p
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

import matplotlib.pyplot as plt
import pandas as pd

from src.evaluation.evaluator import evaluate_golden_set
from src.evaluation.fairness import compute_fairness, fairness_summary_table
from src.evaluation.golden_set import coverage_summary, load_golden_cases
from src.evaluation.sensitivity import run_sensitivity_study

plt.rcParams["figure.dpi"] = 110
print("Projeto:", PROJECT_ROOT)

## Golden set

In [ ]:
cases = load_golden_cases()
print(f"Casos: {len(cases)}")
print("Cobertura:", coverage_summary(cases))

golden = evaluate_golden_set(cases)
print(f"Pass rate: {golden.pass_rate * 100:.1f}%")

rows = [
    {
        "case_id": r.case.case_id,
        "category": r.case.category,
        "predicted": r.predicted_arm_id,
        "expected": r.case.expected_arm_id,
        "passed": r.passed,
    }
    for r in golden.results
]
pd.DataFrame(rows)

## Sensibilidade (subconjunto rapido)

In [ ]:
sensitivity = run_sensitivity_study(horizon=5000, seeds=8)
sens_df = pd.DataFrame([s.__dict__ for s in sensitivity])
sens_df.head(12)

## Fairness de exposicao

In [ ]:
fairness = compute_fairness(max_rows=2000, seed=42)
fairness_summary_table(fairness)

In [ ]:
fig_path = PROJECT_ROOT / "reports" / "figures" / "offline_fairness_exposure.png"
if fig_path.exists():
    plt.imshow(plt.imread(fig_path))
    plt.axis("off")
    plt.title("Fairness: observado vs. guloso (gerado pelo CLI)")
else:
    print("Rode: poetry run python -m src.evaluation")